# Trackinsight export

In [1]:
import json
import os
import time
from pathlib import Path

import polars as pl
import requests
import trackinsight_data_python as API
from dotenv import load_dotenv
from IPython.display import display
from trackinsight_data_python import partitions as ti_partitions

def find_project_root() -> Path:
    """Project root, whether the kernel cwd is code/ or notebooks/."""
    start = Path.cwd().resolve()
    for directory in [start, *start.parents]:
        if (directory / "pyproject.toml").is_file():
            return directory
    raise FileNotFoundError(f"Could not find pyproject.toml, starting from {start}")

root = find_project_root()
env_path = root / ".env"
if not env_path.is_file():
    raise FileNotFoundError(env_path)
load_dotenv(env_path, override=True)

CACHE_DIR = root / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)
STATIC_CACHE_PATH = CACHE_DIR / "cache-trackinsight-static.json"
MONTHLY_CACHE_PATH = CACHE_DIR / "cache-trackinsight-monthly.json"

TOTAL_RETURN_INDEXES = ["SPTR Index", "SPXT Index"]
HISTORY_START = "2010-01-01"
PAUSE_SECONDS = 0.5
MONTHLY_METRICS = ("return", "excess_return", "nav", "tracking_error")
ACTIVE_OBJECTIVE = (
    r"(?i)(benchmark|outperform|relative to).{0,80}s&p\s*500"
    r"|s&p\s*500.{0,40}(benchmark|total return|net return|index)"
)

# Slow the Trackinsight partition downloads and retry after HTTP 429.
os.environ["TRACK_API_DL_WORKERS"] = "2"
_original_get_partition = getattr(ti_partitions, "_get_partition_without_retry", ti_partitions.getPartition)
ti_partitions._get_partition_without_retry = _original_get_partition

def get_partition_with_retry(*args, **kwargs):
    delay = 2.0
    for attempt in range(8):
        try:
            return _original_get_partition(*args, **kwargs)
        except requests.HTTPError as error:
            response = error.response
            status = response.status_code if response is not None else None
            if status not in (429, 500, 503) or attempt == 7:
                raise
            retry_after = response.headers.get("Retry-After") if response is not None else None
            try:
                wait = float(retry_after) if retry_after else delay
            except ValueError:
                wait = delay
            wait = min(max(wait, 1.0), 120.0)
            print(f"Trackinsight HTTP {status}, waiting {wait:.0f}s", flush=True)
            time.sleep(wait)
            delay = min(delay * 2, 60)

ti_partitions.getPartition = get_partition_with_retry

def load_cache(path: Path) -> dict:
    if not path.exists():
        return {}
    try:
        cache = json.loads(path.read_text())
    except json.JSONDecodeError:
        return {}
    return cache if isinstance(cache, dict) else {}

def save_cache(path: Path, cache: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".json.tmp")
    temporary.write_text(json.dumps(cache, indent=2) + "\n")
    temporary.replace(path)

def frame_records(frame: pl.DataFrame | None) -> list[dict]:
    if frame is None or frame.height == 0:
        return []
    return json.loads(frame.write_json())

def list_empty(column: str) -> pl.Expr:
    values = pl.col(column)
    return values.is_null() | (values.list.len() == 0)

def list_excludes(column: str, label: str) -> pl.Expr:
    return ~pl.col(column).list.contains(label).fill_null(False)

def select_sp500(shares: pl.DataFrame) -> pl.DataFrame:
    """Listed US USD unhedged capitalization share classes tied to the S&P 500 screen."""
    list_columns = ("class_leverage", "class_inverse_leverage", "class_strategy")
    shares = shares.with_columns(
        pl.col(column).cast(pl.List(pl.Utf8), strict=False) for column in list_columns
    )
    not_levered = (
        list_empty("class_leverage")
        & list_empty("class_inverse_leverage")
        & list_excludes("class_strategy", "Leveraged")
        & list_excludes("class_strategy", "Inverse")
    )
    passive = shares.filter(
        (~pl.col("actively_managed"))
        & pl.col("listed")
        & (pl.col("generic_index") == "S&P 500 Index")
        & not_levered
    ).with_columns(pl.lit("passive").alias("mgmt_style"))
    active = shares.filter(
        pl.col("actively_managed")
        & pl.col("listed")
        & pl.col("investment_objective").str.contains(ACTIVE_OBJECTIVE)
    ).with_columns(pl.lit("active").alias("mgmt_style"))
    chosen = pl.concat([passive, active], how="diagonal_relaxed")
    return (
        chosen.filter(
            (pl.col("jurisdiction") == "United States")
            & (~pl.col("ucits"))
            & (pl.col("currency") == "USD")
            & (~pl.col("currency_hedged"))
            & (pl.col("dividend_policy") == "Capitalization")
        )
        .with_columns(pl.when(pl.col("mgmt_style") == "passive").then(0).otherwise(1).alias("_order"))
        .sort(["_order", "share_label"])
        .drop("_order")
    )

shares = API.getShares()
passive = select_sp500(shares).filter(
    (pl.col("mgmt_style") == "passive")
    & (pl.col("product_type") == "ETF")
    & pl.col("index_ticker").is_in(TOTAL_RETURN_INDEXES)
)
if passive.height == 0:
    raise RuntimeError("No passive US ETFs matched the S&P 500 Total Return Index")
print(f"{passive.height} passive S&P 500 ETFs")
passive.select("share_id", "main_ticker", "share_label", "index_ticker", "provider").head(10)

loading shares... 100.0% of 20 partitions
4 passive S&P 500 ETFs


share_id,main_ticker,share_label,index_ticker,provider
i32,str,str,str,str
5101,"""SPYM""","""State Street SPDR Portfolio S&…","""SPXT Index""","""State Street Investment Manage…"
4942,"""SPY""","""State Street SPDR S&P 500 ETF …","""SPXT Index""","""State Street Investment Manage…"
4960,"""VOO""","""Vanguard S&P 500 ETF - USD""","""SPTR Index""","""Vanguard"""
4886,"""IVV""","""iShares Core S&P 500 ETF - USD""","""SPTR Index""","""iShares"""


In [2]:
# Static USD report for each passive ETF. One share class at a time, saved after each fetch.
static_cache = load_cache(STATIC_CACHE_PATH)
ids = [int(value) for value in passive["share_id"].to_list()]
labels = {
    int(share_id): (str(ticker).strip() if ticker else f"share_{int(share_id)}")
    for share_id, ticker in passive.select("share_id", "main_ticker").iter_rows()
}
pending = [share_id for share_id in ids if str(share_id) not in static_cache]
print(f"{len(ids) - len(pending)} reports in the cache, {len(pending)} still to fetch")

for index, share_id in enumerate(pending, start=1):
    key = str(share_id)
    label = labels.get(share_id, key)
    print(f"fetching report for {label} ({key})", flush=True)
    rows = frame_records(API.getReports(ccy="usd", ids=[share_id]))
    static_cache[key] = rows[0] if len(rows) == 1 else (rows or None)
    save_cache(STATIC_CACHE_PATH, static_cache)
    if index < len(pending):
        time.sleep(PAUSE_SECONDS)

report_rows = []
for share_id in ids:
    stored = static_cache.get(str(share_id))
    if isinstance(stored, dict):
        report_rows.append(stored)
    elif isinstance(stored, list):
        report_rows.extend(row for row in stored if isinstance(row, dict))
if not report_rows:
    raise RuntimeError("Trackinsight returned no reports for the selected ETFs")

reports = pl.from_dicts(report_rows, infer_schema_length=None)
if "main_ticker" in reports.columns:
    reports = reports.sort("main_ticker")
print(f"{reports.height} reports, {reports.width} columns")
display(reports)


4 reports in the cache, 0 still to fetch
4 reports, 221 columns


share_id,fund_id,product_type,provider,fund_label,share_class,isin,isin_1,isin_2,isin_3,share_label,investment_objective,main_ticker,main_ticker_currency,listing_place,expense_ratio,jurisdiction,zone,fund_structure,ucits,shariah_compliant,sfdr_article,sfdr_article_date,is_semi_transparent,semi_transparent_strategy,inception_date,delisting_date,listed,currency,currency_hedged,dynamic_hedge,dividend_policy,single_stock,actively_managed,replication_method,replication_model,tracking_since,…,ytd_net_excess_return,1ytd_number_of_points,1ytd_to,1ytd_from,1ytd_annualized_net_return,1ytd_usd_flows,1ytd_usd_net_return,1ytd_dividend_yield,1ytd_flows,1ytd_gross_return,1ytd_last_aum,1ytd_last_usd_aum,1ytd_net_return,1ytd_last_nav_per_share,1ytd_last_outstanding_shares,1ytd_annualized_gross_tracking_error,1ytd_annualized_net_tracking_error,1ytd_gross_excess_return,1ytd_net_excess_return,3y_number_of_points,3y_to,3y_from,3y_annualized_net_return,3y_usd_flows,3y_usd_net_return,3y_dividend_yield,3y_flows,3y_gross_return,3y_last_aum,3y_last_usd_aum,3y_net_return,3y_last_nav_per_share,3y_last_outstanding_shares,3y_annualized_gross_tracking_error,3y_annualized_net_tracking_error,3y_gross_excess_return,3y_net_excess_return
i64,i64,str,str,str,str,str,str,null,null,str,str,str,str,str,str,str,str,str,bool,bool,str,null,bool,str,str,null,bool,str,bool,bool,str,null,bool,str,str,str,…,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,f64,f64,f64,f64
4886,334,"""ETF""","""iShares""","""iShares Core S&P 500 ETF""","""USD Share Class""","""US4642872000""","""US4642872000""",null,null,"""iShares Core S&P 500 ETF - USD""","""The iShares Core S&P 500 ETF s…","""IVV""","""USD""","""ARCX""","""0.00030000""","""United States""","""America""","""Open-end Fund""",false,false,"""Out of scope""",null,false,"""Transparent""","""2000-05-15""",null,true,"""USD""",false,false,"""Distribution""",null,false,"""Direct (Physical)""","""Full Replication""","""2000-05-15""",…,-0.000284,241.0,"""2026-10-20""","""2025-10-31""",0.144763,6.6397e10,0.140108,0.010874,6.6397e10,0.140439,8.8577e11,8.8577e11,0.140108,"""772.082388""",1.1472e9,0.000021,0.000021,-0.000029,-0.000361,750.0,"""2026-10-20""","""2023-10-20""",0.237595,2.0250e11,0.896657,0.030557,2.0250e11,0.898363,8.8577e11,8.8577e11,0.896657,"""772.082388""",1.1472e9,0.000037,0.000038,-0.000025,-0.001731
4942,2920,"""ETF""","""State Street Investment Manage…","""State Street SPDR S&P 500 ETF …","""USD Share Class""","""US78462F1030""","""US78462F1030""",null,null,"""State Street SPDR S&P 500 ETF …","""The SPDR® S&P 500® ETF Trust s…","""SPY""","""USD""","""ARCX""","""0.00094500""","""United States""","""America""","""Unit Investment Trust""",false,false,"""Out of scope""",null,false,"""Transparent""","""1993-01-22""",null,true,"""USD""",false,false,"""Distribution""",null,false,"""Direct (Physical)""","""Full Replication""","""1993-01-22""",…,-0.000777,242.0,"""2026-10-20""","""2025-10-31""",0.144114,3.5698e10,0.139482,0.009865,3.5698e10,0.140525,8.1839e11,8.1839e11,0.139482,"""768.668932""",1.0647e9,0.0001924,0.00019,0.000057,-0.000987,751.0,"""2026-10-20""","""2023-10-20""",0.236589,7.8896e10,0.892032,0.028415,7.8896e10,0.897398,8.1839e11,8.1839e11,0.892032,"""768.668932""",1.0647e9,0.000215,0.000211,-0.000991,-0.006357
5101,2897,"""ETF""","""State Street Investment Manage…","""State Street SPDR Portfolio S&…","""USD Share Class""","""US78464A8541""","""US78464A8541""",null,null,"""State Street SPDR Portfolio S&…","""The State Street SPDR Portfoli…","""SPYM""","""USD""","""ARCX""","""0.00020000""","""United States""","""America""","""Open-end Fund""",false,false,"""Out of scope""",null,false,"""Transparent""","""2005-11-08""",null,true,"""USD""",false,false,"""Distribution""",null,false,"""Direct (Physical)""","""Optimized Sampling""","""2020-01-24""",…,-0.000264,242.0,"""2026-10-20""","""2025-10-31""",0.144808,6.9976e10,0.140152,0.010065

In [3]:
# Monthly history for each passive ETF, then a month-labelled panel.
# One share class at a time, saved after each fetch.

def share_tickers(selected: pl.DataFrame) -> pl.DataFrame:
    used: dict[str, int] = {}
    rows = []
    for share_id, ticker in selected.select("share_id", "main_ticker").iter_rows():
        base = str(ticker).strip() if ticker else ""
        base = base or f"share_{int(share_id)}"
        seen = used.get(base, 0)
        used[base] = seen + 1
        name = base if seen == 0 else f"{base}_{int(share_id)}"
        rows.append({"id": int(share_id), "ticker": name})
    return pl.DataFrame(rows).with_columns(pl.col("id").cast(pl.Int32))

def monthly_piece(stored: list, share_id: int) -> pl.DataFrame:
    """Normalize one monthly series. Trackinsight uses from/to and last_nav_per_share."""
    piece = pl.from_dicts(stored, infer_schema_length=None)
    date_column = "from" if "from" in piece.columns else "date"
    nav_column = "last_nav_per_share" if "last_nav_per_share" in piece.columns else "nav_per_share"
    piece = piece.with_columns(
        pl.lit(int(share_id)).cast(pl.Int32).alias("id"),
        pl.col(date_column).cast(pl.Utf8).str.slice(0, 10).str.to_date().alias("date"),
        pl.col(nav_column).alias("nav_per_share") if nav_column in piece.columns else pl.lit(None).cast(pl.Float64).alias("nav_per_share"),
    )
    for column in ("net_return", "net_excess_return", "nav_per_share"):
        if column in piece.columns:
            piece = piece.with_columns(pl.col(column).cast(pl.Float64, strict=False))
        else:
            piece = piece.with_columns(pl.lit(None).cast(pl.Float64).alias(column))
    return piece.select("id", "date", "net_return", "net_excess_return", "nav_per_share")

def monthly_by_ticker(monthly: pl.DataFrame, tickers: pl.DataFrame) -> pl.DataFrame:
    """One row per calendar month. Trackinsight already publishes monthly returns."""
    frame = (
        monthly.sort(["id", "date"])
        .with_columns(pl.col("date").dt.truncate("1mo").alias("month"))
        .group_by(["month", "id"], maintain_order=True)
        .agg(
            pl.col("net_return").drop_nulls().last().alias("return"),
            pl.col("net_excess_return").drop_nulls().last().alias("excess_return"),
            pl.col("nav_per_share").drop_nulls().last().alias("nav"),
            # One observation per month, so within-month tracking error is undefined.
            pl.lit(None).cast(pl.Float64).alias("tracking_error"),
        )
        .join(tickers, on="id", how="inner")
    )
    pieces = []
    for metric in MONTHLY_METRICS:
        wide = frame.pivot(on="ticker", index="month", values=metric).sort("month")
        pieces.append(
            wide.rename({column: f"{column}__{metric}" for column in wide.columns if column != "month"})
        )
    panel = pieces[0]
    for piece in pieces[1:]:
        panel = panel.join(piece, on="month", how="full", coalesce=True)
    ordered = ["month"]
    for ticker in sorted(tickers["ticker"].to_list()):
        ordered.extend(f"{ticker}__{metric}" for metric in MONTHLY_METRICS)
    return panel.select([column for column in ordered if column in panel.columns]).sort("month")

monthly_cache = load_cache(MONTHLY_CACHE_PATH)
ids = [int(value) for value in passive["share_id"].to_list()]
labels = {
    int(share_id): (str(ticker).strip() if ticker else f"share_{int(share_id)}")
    for share_id, ticker in passive.select("share_id", "main_ticker").iter_rows()
}
pending = [share_id for share_id in ids if str(share_id) not in monthly_cache]
print(f"{len(ids) - len(pending)} monthly series in the cache, {len(pending)} still to fetch")

for index, share_id in enumerate(pending, start=1):
    key = str(share_id)
    label = labels.get(share_id, key)
    print(f"fetching monthly timeseries for {label} ({key})", flush=True)
    monthly = API.getMonthlyTimeseries(start=HISTORY_START, end=None, ccy="usd", ids=[share_id])
    if monthly is None or monthly.height == 0:
        monthly_cache[key] = None
    else:
        keep = [column for column in monthly.columns if column != "id"]
        monthly_cache[key] = frame_records(monthly.select(keep))
    save_cache(MONTHLY_CACHE_PATH, monthly_cache)
    if index < len(pending):
        time.sleep(PAUSE_SECONDS)

pieces = []
for share_id in ids:
    stored = monthly_cache.get(str(share_id))
    if not isinstance(stored, list) or not stored:
        continue
    pieces.append(monthly_piece(stored, share_id))
if not pieces:
    raise RuntimeError("Trackinsight returned no monthly timeseries for the selected ETFs")

monthly = pl.concat(pieces, how="vertical_relaxed")
passive_monthly = monthly_by_ticker(monthly, share_tickers(passive))
print(f"monthly series {passive_monthly.shape[0]} months x {passive_monthly.width - 1} columns")
passive_monthly

4 monthly series in the cache, 0 still to fetch
monthly series 118 months x 16 columns


month,IVV__return,IVV__excess_return,IVV__nav,IVV__tracking_error,SPY__return,SPY__excess_return,SPY__nav,SPY__tracking_error,SPYM__return,SPYM__excess_return,SPYM__nav,SPYM__tracking_error,VOO__return,VOO__excess_return,VOO__nav,VOO__tracking_error
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2017-01-01,0.018919,-0.000047,228.89,null,0.018932,-0.000034,227.5107,null,0.019975,null,106.9116,null,0.018927,-0.00004,208.88,null
2017-02-01,0.039713,0.000008,237.98,null,0.039551,-0.000157,236.509,null,0.038557,null,111.0338,null,0.03964,-0.000066,217.16,null
2017-03-01,0.001137,-0.000029,237.21,null,0.001049,-0.000116,235.7303,null,-0.002607,null,110.6282,null,0.001125,-0.000042,216.4,null
2017-04-01,0.010244,-0.000026,239.64,null,0.010199,-0.000072,238.1345,null,0.010469,null,111.7864,null,0.010213,-0.000057,218.61,null
2017-05-01,0.014021,-0.000052,243.0,null,0.014009,-0.000063,241.4706,null,0.012698,null,113.2059,null,0.014043,-0.000029,221.68,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-06-01,-0.009556,-0.000032,750.616073,null,-0.009579,-0.000056,747.13297,null,-0.00956,-0.000037,87.933447,null,-0.009548,-0.000024,686.7,null
2026-07-01,-0.000663,-0.000028,750.118576,null,-0.000708,-0.000073,746.604287,null,-0.000647,-0.000012,87.87659,null,-0.00067,-0.000035,686.24,null
2026-08-01,0.027205,-0.000029,770.525507,null,0.02713,-0.000103,766.859415,null,0.027207,-0.000026,90.267479,null,0.027221,-0.000013,704.92,null


In [4]:
# Active ETFs are not tied to an index ticker the way passive ones are.
# select_sp500 already keeps listed, USD, unhedged, capitalization US share classes.
# Here we keep ETFs whose objective benchmarks or targets the S&P 500
# (including the total-return index).
active = select_sp500(shares).filter(
    (pl.col("mgmt_style") == "active")
    & (pl.col("product_type") == "ETF")
)
if active.height == 0:
    raise RuntimeError("No active US ETFs matched the S&P 500 screen")
print(f"{active.height} active S&P 500 ETFs")
active.select(
    "share_id",
    "main_ticker",
    "share_label",
    "benchmark",
    "index_ticker",
    "provider",
    "investment_objective",
).head(20)


69 active S&P 500 ETFs


share_id,main_ticker,share_label,benchmark,index_ticker,provider,investment_objective
i32,str,str,str,str,str,str
25301,"""RMRC""","""ARMOR Core Risk-Managed ETF - …",null,null,"""Exchange Traded Concepts""","""The Fund is actively managed a…"
13149,"""VEGA""","""AdvisorShares STAR Global Buy-…",null,null,"""AdvisorShares""","""The AdvisorShares STAR Global …"
16202,"""SIXO""","""AllianzIM U.S. Large Cap 6 Mon…",null,null,"""Allianz Investment Management …","""The AllianzIM U.S. Large Cap 6…"
16905,"""SIXJ""","""AllianzIM U.S. Large Cap 6 Mon…",null,null,"""Allianz Investment Management …","""The Fund seeks to match the re…"
13727,"""APRT""","""AllianzIM U.S. Large Cap Buffe…",null,null,"""Allianz Investment Management …","""The AllianzIM U.S. Large Cap B…"
…,…,…,…,…,…,…
13856,"""SSUS""","""Day Hagan/Ned Davis Research S…",null,null,"""Day Hagan Asset Management""","""The fund seeks to enhance retu…"
21061,"""WDTE""","""Defiance S&P 500 Enhanced Opti…",null,null,"""Defiance ETFs""","""The Fund’s primary investment …"
20213,"""SPYT""","""Defiance S&P 500 Target Income…",null,null,"""Defiance ETFs""","""The Fund’s primary investment …"


In [5]:
# Static USD report for each active ETF. Same cache file as the passive reports.
# One share class at a time, saved after each fetch.
static_cache = load_cache(STATIC_CACHE_PATH)
ids = [int(value) for value in active["share_id"].to_list()]
labels = {
    int(share_id): (str(ticker).strip() if ticker else f"share_{int(share_id)}")
    for share_id, ticker in active.select("share_id", "main_ticker").iter_rows()
}
pending = [share_id for share_id in ids if str(share_id) not in static_cache]
print(f"{len(ids) - len(pending)} reports in the cache, {len(pending)} still to fetch")

for index, share_id in enumerate(pending, start=1):
    key = str(share_id)
    label = labels.get(share_id, key)
    print(f"fetching report for {label} ({key})", flush=True)
    rows = frame_records(API.getReports(ccy="usd", ids=[share_id]))
    static_cache[key] = rows[0] if len(rows) == 1 else (rows or None)
    save_cache(STATIC_CACHE_PATH, static_cache)
    if index < len(pending):
        time.sleep(PAUSE_SECONDS)

report_rows = []
for share_id in ids:
    stored = static_cache.get(str(share_id))
    if isinstance(stored, dict):
        report_rows.append(stored)
    elif isinstance(stored, list):
        report_rows.extend(row for row in stored if isinstance(row, dict))
if not report_rows:
    raise RuntimeError("Trackinsight returned no reports for the selected ETFs")

active_reports = pl.from_dicts(report_rows, infer_schema_length=None)
if "main_ticker" in active_reports.columns:
    active_reports = active_reports.sort("main_ticker")
print(f"{active_reports.height} reports, {active_reports.width} columns")
display(active_reports)
active_reports


68 reports in the cache, 1 still to fetch
fetching report for FIFT (27702)
loading reports... 100.0% of 1 partitions
69 reports, 221 columns


share_id,fund_id,product_type,provider,fund_label,share_class,isin,isin_1,isin_2,isin_3,share_label,investment_objective,main_ticker,main_ticker_currency,listing_place,expense_ratio,jurisdiction,zone,fund_structure,ucits,shariah_compliant,sfdr_article,sfdr_article_date,is_semi_transparent,semi_transparent_strategy,inception_date,delisting_date,listed,currency,currency_hedged,dynamic_hedge,dividend_policy,single_stock,actively_managed,replication_method,replication_model,tracking_since,…,ytd_net_excess_return,1ytd_number_of_points,1ytd_to,1ytd_from,1ytd_annualized_net_return,1ytd_usd_flows,1ytd_usd_net_return,1ytd_dividend_yield,1ytd_flows,1ytd_gross_return,1ytd_last_aum,1ytd_last_usd_aum,1ytd_net_return,1ytd_last_nav_per_share,1ytd_last_outstanding_shares,1ytd_annualized_gross_tracking_error,1ytd_annualized_net_tracking_error,1ytd_gross_excess_return,1ytd_net_excess_return,3y_number_of_points,3y_to,3y_from,3y_annualized_net_return,3y_usd_flows,3y_usd_net_return,3y_dividend_yield,3y_flows,3y_gross_return,3y_last_aum,3y_last_usd_aum,3y_net_return,3y_last_nav_per_share,3y_last_outstanding_shares,3y_annualized_gross_tracking_error,3y_annualized_net_tracking_error,3y_gross_excess_return,3y_net_excess_return
i64,i64,str,str,str,str,str,str,null,null,str,str,str,str,str,str,str,str,str,bool,bool,str,null,bool,str,str,null,bool,str,bool,bool,str,null,bool,null,null,null,…,null,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,null,null,null,null,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,null,null,null,null
13727,7724,"""ETF""","""Allianz Investment Management …","""AllianzIM U.S. Large Cap Buffe…","""C - USD Share Class""","""US00888H1095""","""US00888H1095""",null,null,"""AllianzIM U.S. Large Cap Buffe…","""The AllianzIM U.S. Large Cap B…","""APRT""","""USD""","""BATS""","""0.00740000""","""United States""","""America""","""Open-end Fund""",false,false,"""Out of scope""",null,false,"""Transparent""","""2020-05-28""",null,true,"""USD""",false,false,"""Distribution""",null,true,null,null,null,…,null,240.0,"""2026-10-19""","""2025-10-31""",0.157599,226738.0,0.152042,0.0,226738.0,0.16031,4.72107e7,4.72107e7,0.152042,null,null,null,null,null,null,749.0,"""2026-10-19""","""2023-10-20""",0.157152,-2.472586e6,0.549426,0.0,-2.472586e6,0.584152,4.72107e7,4.72107e7,0.549426,null,null,null,null,null,null
13728,7725,"""ETF""","""Allianz Investment Management …","""AllianzIM U.S. Large Cap Buffe…","""C - USD Share Class""","""US00888H2085""","""US00888H2085""",null,null,"""AllianzIM U.S. Large Cap Buffe…","""The AllianzIM U.S. Large Cap B…","""APRW""","""USD""","""BATS""","""0.00740000""","""United States""","""America""","""Open-end Fund""",false,false,"""Out of scope""",null,false,"""Transparent""","""2020-05-28""",null,true,"""USD""",false,false,"""Distribution""",null,true,null,null,null,…,null,242.0,"""2026-10-19""","""2025-10-31""",0.107786,1.5567216e7,0.104064,0.0,1.5567216e7,0.111989,2.03343849e8,2.03343849e8,0.104064,null,null,null,null,null,null,751.0,"""2026-10-19""","""2023-10-20""",0.109296,1.5103791e7,0.365031,0.0,1.5103791e7,0.39563,2.03343849e8,2.03343849e8,0.365031,null,null,null,null,null,null
23625,16153,"""ETF""","""TrueShares""","""TrueShares Structured Outcome …","""D - USD Share Class""","""US2103228555""","""US2103228555""",null,null,"""TrueShares Structured Outcome …","""The TrueShares Structured Outc…","""APRZ""","""USD""","""BATS""","""0.00790000""","""United States""","""America""","""Open-end Fund""",false,false,"""Out of scope""",null,false,"""Transparent""","""2021-04-01""",null,true,"""USD""",false,false,"""Distribution""",null,true,null,null,null,…,null,241.0,"""2026-10-20""","""2025-10-31""",0.08677,1.6458448e7,0.084049,0.030892,1.6458448e7,0.092381,3.3301348e7,3.3301348e7,0.084049,"""40.611400""",820000.0,null,null,null,null,747.0,"""2026-10-20""","""2023-10-20""",0.167801,2.2567482e7,0.593278,0.075436,2.2567482e7,0.63146,3.3301348e7,3.3301348e7,0.593278,"""40.611400""",820000.0,null,null,null

share_id,fund_id,product_type,provider,fund_label,share_class,isin,isin_1,isin_2,isin_3,share_label,investment_objective,main_ticker,main_ticker_currency,listing_place,expense_ratio,jurisdiction,zone,fund_structure,ucits,shariah_compliant,sfdr_article,sfdr_article_date,is_semi_transparent,semi_transparent_strategy,inception_date,delisting_date,listed,currency,currency_hedged,dynamic_hedge,dividend_policy,single_stock,actively_managed,replication_method,replication_model,tracking_since,…,ytd_net_excess_return,1ytd_number_of_points,1ytd_to,1ytd_from,1ytd_annualized_net_return,1ytd_usd_flows,1ytd_usd_net_return,1ytd_dividend_yield,1ytd_flows,1ytd_gross_return,1ytd_last_aum,1ytd_last_usd_aum,1ytd_net_return,1ytd_last_nav_per_share,1ytd_last_outstanding_shares,1ytd_annualized_gross_tracking_error,1ytd_annualized_net_tracking_error,1ytd_gross_excess_return,1ytd_net_excess_return,3y_number_of_points,3y_to,3y_from,3y_annualized_net_return,3y_usd_flows,3y_usd_net_return,3y_dividend_yield,3y_flows,3y_gross_return,3y_last_aum,3y_last_usd_aum,3y_net_return,3y_last_nav_per_share,3y_last_outstanding_shares,3y_annualized_gross_tracking_error,3y_annualized_net_tracking_error,3y_gross_excess_return,3y_net_excess_return
i64,i64,str,str,str,str,str,str,null,null,str,str,str,str,str,str,str,str,str,bool,bool,str,null,bool,str,str,null,bool,str,bool,bool,str,null,bool,null,null,null,…,null,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,null,null,null,null,f64,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,str,f64,null,null,null,null
13727,7724,"""ETF""","""Allianz Investment Management …","""AllianzIM U.S. Large Cap Buffe…","""C - USD Share Class""","""US00888H1095""","""US00888H1095""",null,null,"""AllianzIM U.S. Large Cap Buffe…","""The AllianzIM U.S. Large Cap B…","""APRT""","""USD""","""BATS""","""0.00740000""","""United States""","""America""","""Open-end Fund""",false,false,"""Out of scope""",null,false,"""Transparent""","""2020-05-28""",null,true,"""USD""",false,false,"""Distribution""",null,true,null,null,null,…,null,240.0,"""2026-10-19""","""2025-10-31""",0.157599,226738.0,0.152042,0.0,226738.0,0.16031,4.72107e7,4.72107e7,0.152042,null,null,null,null,null,null,749.0,"""2026-10-19""","""2023-10-20""",0.157152,-2.472586e6,0.549426,0.0,-2.472586e6,0.584152,4.72107e7,4.72107e7,0.549426,null,null,null,null,null,null
13728,7725,"""ETF""","""Allianz Investment Management …","""AllianzIM U.S. Large Cap Buffe…","""C - USD Share Class""","""US00888H2085""","""US00888H2085""",null,null,"""AllianzIM U.S. Large Cap Buffe…","""The AllianzIM U.S. Large Cap B…","""APRW""","""USD""","""BATS""","""0.00740000""","""United States""","""America""","""Open-end Fund""",false,false,"""Out of scope""",null,false,"""Transparent""","""2020-05-28""",null,true,"""USD""",false,false,"""Distribution""",null,true,null,null,null,…,null,242.0,"""2026-10-19""","""2025-10-31""",0.107786,1.5567216e7,0.104064,0.0,1.5567216e7,0.111989,2.03343849e8,2.03343849e8,0.104064,null,null,null,null,null,null,751.0,"""2026-10-19""","""2023-10-20""",0.109296,1.5103791e7,0.365031,0.0,1.5103791e7,0.39563,2.03343849e8,2.03343849e8,0.365031,null,null,null,null,null,null
23625,16153,"""ETF""","""TrueShares""","""TrueShares Structured Outcome …","""D - USD Share Class""","""US2103228555""","""US2103228555""",null,null,"""TrueShares Structured Outcome …","""The TrueShares Structured Outc…","""APRZ""","""USD""","""BATS""","""0.00790000""","""United States""","""America""","""Open-end Fund""",false,false,"""Out of scope""",null,false,"""Transparent""","""2021-04-01""",null,true,"""USD""",false,false,"""Distribution""",null,true,null,null,null,…,null,241.0,"""2026-10-20""","""2025-10-31""",0.08677,1.6458448e7,0.084049,0.030892,1.6458448e7,0.092381,3.3301348e7,3.3301348e7,0.084049,"""40.611400""",820000.0,null,null,null,null,747.0,"""2026-10-20""","""2023-10-20""",0.167801,2.2567482e7,0.593278,0.075436,2.2567482e7,0.63146,3.3301348e7,3.3301348e7,0.593278,"""40.611400""",820000.0,null,null,null

In [6]:
# Monthly history for each active ETF, then a month-labelled panel.
# Same cache file as the passive monthly series. One share class at a time, saved after each fetch.
monthly_cache = load_cache(MONTHLY_CACHE_PATH)
ids = [int(value) for value in active["share_id"].to_list()]
labels = {
    int(share_id): (str(ticker).strip() if ticker else f"share_{int(share_id)}")
    for share_id, ticker in active.select("share_id", "main_ticker").iter_rows()
}
pending = [share_id for share_id in ids if str(share_id) not in monthly_cache]
print(f"{len(ids) - len(pending)} monthly series in the cache, {len(pending)} still to fetch")

for index, share_id in enumerate(pending, start=1):
    key = str(share_id)
    label = labels.get(share_id, key)
    print(f"fetching monthly timeseries for {label} ({key})", flush=True)
    monthly = API.getMonthlyTimeseries(start=HISTORY_START, end=None, ccy="usd", ids=[share_id])
    if monthly is None or monthly.height == 0:
        monthly_cache[key] = None
    else:
        keep = [column for column in monthly.columns if column != "id"]
        monthly_cache[key] = frame_records(monthly.select(keep))
    save_cache(MONTHLY_CACHE_PATH, monthly_cache)
    if index < len(pending):
        time.sleep(PAUSE_SECONDS)

pieces = []
for share_id in ids:
    stored = monthly_cache.get(str(share_id))
    if not isinstance(stored, list) or not stored:
        continue
    pieces.append(monthly_piece(stored, share_id))
if not pieces:
    raise RuntimeError("Trackinsight returned no monthly timeseries for the selected ETFs")

active_monthly_rows = pl.concat(pieces, how="vertical_relaxed")
active_monthly = monthly_by_ticker(active_monthly_rows, share_tickers(active))
print(f"monthly series {active_monthly.shape[0]} months x {active_monthly.width - 1} columns")
active_monthly

68 monthly series in the cache, 1 still to fetch
fetching monthly timeseries for FIFT (27702)
loading monthly_timeseries... 100.0% of 10 partitions
monthly series 118 months x 276 columns


month,APRT__return,APRT__excess_return,APRT__nav,APRT__tracking_error,APRW__return,APRW__excess_return,APRW__nav,APRW__tracking_error,APRZ__return,APRZ__excess_return,APRZ__nav,APRZ__tracking_error,AUGZ__return,AUGZ__excess_return,AUGZ__nav,AUGZ__tracking_error,BFIX__return,BFIX__excess_return,BFIX__nav,BFIX__tracking_error,BUIL__return,BUIL__excess_return,BUIL__nav,BUIL__tracking_error,CAGE__return,CAGE__excess_return,CAGE__nav,CAGE__tracking_error,CAPE__return,CAPE__excess_return,CAPE__nav,CAPE__tracking_error,DECZ__return,DECZ__excess_return,DECZ__nav,DECZ__tracking_error,…,UXJA__tracking_error,UXJL__return,UXJL__excess_return,UXJL__nav,UXJL__tracking_error,UXOC__return,UXOC__excess_return,UXOC__nav,UXOC__tracking_error,VEGA__return,VEGA__excess_return,VEGA__nav,VEGA__tracking_error,VOOB__return,VOOB__excess_return,VOOB__nav,VOOB__tracking_error,WDTE__return,WDTE__excess_return,WDTE__nav,WDTE__tracking_error,XDIV__return,XDIV__excess_return,XDIV__nav,XDIV__tracking_error,XLSR__return,XLSR__excess_return,XLSR__nav,XLSR__tracking_error,XUSP__return,XUSP__excess_return,XUSP__nav,XUSP__tracking_error,XX__return,XX__excess_return,XX__nav,XX__tracking_error
date,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2017-01-01,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,0.012626,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2017-02-01,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,0.020743,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2017-03-01,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,0.002646,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2017-04-01,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,0.008242,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
2017-05-01,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,0.006679,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
2026-06-01,0.000714,null,null,null,0.00127,null,null,null,-0.010284,null,39.8726,null,-0.012236,null,45.3196,null,-0.004539,null,25.0661,null,null,null,null,null,-0.022325,null,null,null,0.007465,null,null,null,-0.010394,null,43.2061,null,…,null,-0.015595,null,36.362073,null,-0.015535,null,39.100892,null,-0.002412,null,null,null,null,null,null,null,-0.011747,null,29.9491,null,-0.011743,null,30.4154,null,-0.020435,null,64.833632,null,-0.021267,null,52.5652,null,nul

In [7]:
import time

import pandas as pd
import polars as pl

# One Fund sheet for every passive and active ETF, mapped from the static cache.
OUTPUT_PATH = root / "output-data" / "trackinsight-sp500-funds.xlsx"
FUND_HEADERS = [
    "Name",
    "ISIN",
    "Hyper-Classification Scheme",
    "Instrument Type",
    "Domicile",
    "Country Registered for Sale",
    "Currency",
    "Exchange",
    "TIC",
    "Benchmark Name",
    "Main Index",
    "Organization Central Code",
    "Instrument Is Active Flag",
    "Track Exposure Factor",
    "Fund Total Assets under Management",
    "Fund Turnover %",
    "Total Expense Ratio",
    "Fund Inception Date",
    "Performance for 12 Years to Last Month End (%)",
    "Performance for 10 Years to Last Month End",
    "Alpha for 10 Years to Last Month End",
    "Beta for 10 Years to Last Month End",
    "R-Squared for 10 Years to Last Month End",
    "Standard Deviation for 10 Years to Last Month End",
    "Sharpe Ratio for 10 Years to Last Month End",
    "Jensen's Alpha for 10 Years to Last Month End",
    "Performance for 5 Years to Last Month End",
    "Alpha for 5 Years to Last Month End",
    "Beta for 5 Years to Last Month End",
    "R-Squared for 5 Years to Last Month End",
    "Standard Deviation for 5 Years to Last Month End",
    "Sharpe Ratio for 5 Years to Last Month End",
    "Jensen's Alpha for 5 Years to Last Month End",
    "Performance for 3 Years to Last Month End",
    "Alpha for 3 Years to Last Month End",
    "Beta for 3 Years to Last Month End",
    "R-Squared for 3 Years to Last Month End",
    "Standard Deviation for 3 Years to Last Month End",
    "Sharpe Ratio for 3 Years to Last Month End",
    "Jensen's Alpha for 3 Years to Last Month End",
    "Return/Risk Ratio for 3 Years to Last Month End",
    "Tracking Error for 10 Years to Last Month End",
    "Tracking Error for 5 Years to Last Month End",
    "Tracking Error for 3 Years to Last Month End",
    "Tracking Error for 1 Year to Last Month End",
    "Tracking Error for 6 Months to Last Month End",
    "fund_type",
    "fund_wrapper",
]

def first_value(report: dict, *keys):
    for key in keys:
        value = report.get(key)
        if value is None:
            continue
        if isinstance(value, str) and not value.strip():
            continue
        if isinstance(value, (list, tuple, dict)) and len(value) == 0:
            continue
        return value
    return None

def join_countries(value):
    if value is None:
        return None
    if isinstance(value, list):
        parts = []
        for item in value:
            if isinstance(item, dict):
                part = item.get("key") or item.get("name") or item.get("country")
                if part:
                    parts.append(str(part))
            elif item is not None and str(item).strip():
                parts.append(str(item).strip())
        return ", ".join(parts) if parts else None
    return str(value)

def fund_row(report: dict | None, fund_type: str) -> dict:
    report = report if isinstance(report, dict) else {}
    row = {header: None for header in FUND_HEADERS}
    row["Name"] = first_value(report, "fund_label", "share_label")
    row["ISIN"] = first_value(report, "isin")
    row["Instrument Type"] = first_value(report, "product_type")
    row["Domicile"] = first_value(report, "jurisdiction")
    row["Country Registered for Sale"] = join_countries(report.get("distribution_countries"))
    row["Currency"] = first_value(report, "currency")
    row["Exchange"] = first_value(report, "listing_place")
    row["TIC"] = first_value(report, "main_ticker")
    row["Benchmark Name"] = first_value(report, "benchmark")
    row["Main Index"] = first_value(report, "generic_index")
    active_flag = first_value(report, "actively_managed")
    if active_flag is None:
        row["Instrument Is Active Flag"] = None
    else:
        row["Instrument Is Active Flag"] = "Y" if bool(active_flag) else "N"
    row["Fund Total Assets under Management"] = first_value(
        report, "1d_last_usd_aum", "ytd_last_usd_aum"
    )
    expense = first_value(report, "expense_ratio")
    if expense is None:
        row["Total Expense Ratio"] = None
    else:
        # Trackinsight stores the ratio as a fraction (e.g. 0.0002); Lipper uses percent.
        row["Total Expense Ratio"] = float(expense) * 100
    row["Fund Inception Date"] = first_value(report, "inception_date")
    row["Performance for 3 Years to Last Month End"] = first_value(report, "3y_net_return")
    row["Tracking Error for 3 Years to Last Month End"] = first_value(
        report, "3y_annualized_net_tracking_error"
    )
    row["Tracking Error for 1 Year to Last Month End"] = first_value(
        report, "1ytd_annualized_net_tracking_error"
    )
    row["fund_type"] = fund_type
    row["fund_wrapper"] = "ETF"
    return row

static_cache = load_cache(STATIC_CACHE_PATH)
selections = [
    (passive, "Passive"),
    (active, "Active"),
]
rows = []
for selected, fund_type in selections:
    for share_id in selected["share_id"].to_list():
        key = str(int(share_id))
        stored = static_cache.get(key)
        if isinstance(stored, list):
            report = stored[0] if stored and isinstance(stored[0], dict) else None
        elif isinstance(stored, dict):
            report = stored
        else:
            report = None
        rows.append(fund_row(report, fund_type))

fund_sheet = pd.DataFrame(rows, columns=FUND_HEADERS)

# Timeseries sheet: two header rows (ISIN, then "return"), monthly net_return only.
# Trackinsight net_return is the after-fee NAV return; gross_return is before fees.
# For ETFs these are total-return style series (NAV path with distributions reinvested).
from datetime import date
from dateutil.relativedelta import relativedelta

TIMESERIES_START = date(2010, 1, 1)
monthly_cache = load_cache(MONTHLY_CACHE_PATH)

def static_report_for(share_id: int) -> dict:
    stored = static_cache.get(str(int(share_id)))
    if isinstance(stored, list):
        return stored[0] if stored and isinstance(stored[0], dict) else {}
    if isinstance(stored, dict):
        return stored
    return {}

# Fetch any missing monthly series into the cache (needed after HISTORY_START moved to 2010).
share_ids = [int(share_id) for selected, _ in selections for share_id in selected["share_id"].to_list()]
labels = {}
for selected, _ in selections:
    for share_id, ticker in selected.select("share_id", "main_ticker").iter_rows():
        labels[int(share_id)] = str(ticker).strip() if ticker else f"share_{int(share_id)}"
pending = [share_id for share_id in share_ids if str(share_id) not in monthly_cache]
print(f"{len(share_ids) - len(pending)} monthly series in the cache, {len(pending)} still to fetch")
for index, share_id in enumerate(pending, start=1):
    key = str(share_id)
    label = labels.get(share_id, key)
    print(f"fetching monthly timeseries for {label} ({key})", flush=True)
    monthly = API.getMonthlyTimeseries(start=HISTORY_START, end=None, ccy="usd", ids=[share_id])
    if monthly is None or monthly.height == 0:
        monthly_cache[key] = None
    else:
        keep = [column for column in monthly.columns if column != "id"]
        monthly_cache[key] = frame_records(monthly.select(keep))
    save_cache(MONTHLY_CACHE_PATH, monthly_cache)
    if index < len(pending):
        time.sleep(PAUSE_SECONDS)

isin_order = []
long_rows = []
for selected, _fund_type in selections:
    for share_id in selected["share_id"].to_list():
        report = static_report_for(int(share_id))
        isin = first_value(report, "isin") or f"share_{int(share_id)}"
        if isin not in isin_order:
            isin_order.append(isin)
        stored = monthly_cache.get(str(int(share_id)))
        if not isinstance(stored, list) or not stored:
            continue
        piece = monthly_piece(stored, int(share_id)).filter(pl.col("date") >= TIMESERIES_START)
        for month_date, net_return in piece.select("date", "net_return").iter_rows():
            long_rows.append({"date": month_date, "isin": isin, "return": net_return})

if not long_rows:
    raise RuntimeError("No monthly timeseries in the cache for the selected ETFs")

returns = pd.DataFrame(long_rows).pivot(index="date", columns="isin", values="return")
returns = returns.reindex(columns=isin_order)
last_month = returns.index.max()
month_index = []
cursor = TIMESERIES_START
while cursor <= last_month:
    month_index.append(cursor)
    cursor += relativedelta(months=1)
returns = returns.reindex(month_index)

# All fields sheet: every Trackinsight static column, one row per ETF.
def excel_cell(value):
    if value is None:
        return None
    if isinstance(value, (list, dict)):
        return json.dumps(value)
    return value

all_field_rows = []
all_keys = set()
for selected, fund_type in selections:
    for share_id in selected["share_id"].to_list():
        stored = static_cache.get(str(int(share_id)))
        if isinstance(stored, list):
            report = stored[0] if stored and isinstance(stored[0], dict) else {}
        elif isinstance(stored, dict):
            report = stored
        else:
            report = {}
        if not isinstance(report, dict):
            report = {}
        all_keys.update(report.keys())
        row = {key: excel_cell(report.get(key)) for key in report}
        row["fund_type"] = fund_type
        row["fund_wrapper"] = "ETF"
        all_field_rows.append(row)

field_columns = ["fund_type", "fund_wrapper", *sorted(all_keys)]
all_fields_sheet = pd.DataFrame(all_field_rows, columns=field_columns)

OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
with pd.ExcelWriter(OUTPUT_PATH, engine="openpyxl") as writer:
    fund_sheet.to_excel(writer, sheet_name="Fund", index=False)
    all_fields_sheet.to_excel(writer, sheet_name="All fields", index=False)
    timeseries_ws = writer.book.create_sheet("Timeseries", 1)
    timeseries_ws.cell(1, 1, value=None)
    timeseries_ws.cell(2, 1, value="date")
    for column_index, isin in enumerate(isin_order, start=2):
        timeseries_ws.cell(1, column_index, value=isin)
        timeseries_ws.cell(2, column_index, value="return")
    for row_offset, (month_date, row) in enumerate(returns.iterrows(), start=3):
        timeseries_ws.cell(row_offset, 1, value=month_date.strftime("%d/%m/%Y"))
        for column_index, isin in enumerate(isin_order, start=2):
            value = row.get(isin)
            timeseries_ws.cell(
                row_offset,
                column_index,
                value=None if pd.isna(value) else float(value),
            )
print(
    f"wrote {OUTPUT_PATH} "
    f"({len(fund_sheet)} funds, timeseries {len(returns)} months x "
    f"{len(isin_order)} return series from {TIMESERIES_START.strftime('%d/%m/%Y')}, "
    f"all fields {all_fields_sheet.shape[1]} columns)"
)
fund_sheet.head()


73 monthly series in the cache, 0 still to fetch
wrote /Users/olivierdedecker/Documents/02 - Enfants/Alix/Mémoire Alix/data/code/output-data/trackinsight-sp500-funds.xlsx (73 funds, timeseries 202 months x 73 return series from 01/01/2010, all fields 223 columns)


,Name,ISIN,Hyper-Classification Scheme,Instrument Type,Domicile,Country Registered for Sale,Currency,Exchange,TIC,Benchmark Name,...,Sharpe Ratio for 3 Years to Last Month End,Jensen's Alpha for 3 Years to Last Month End,Return/Risk Ratio for 3 Years to Last Month End,Tracking Error for 10 Years to Last Month End,Tracking Error for 5 Years to Last Month End,Tracking Error for 3 Years to Last Month End,Tracking Error for 1 Year to Last Month End,Tracking Error for 6 Months to Last Month End,fund_type,fund_wrapper
0,State Street SPDR Portfolio S&P 500 ETF,US78464A8541,None,ETF,United States,United States,USD,ARCX,SPYM,S&P 500 Total Return Index - USD,...,None,None,None,None,None,0.000115,0.000051,None,Passive,ETF
1,State Street SPDR S&P 500 ETF Trust,US78462F1030,None,ETF,United States,United States,USD,ARCX,SPY,S&P 500 Total Return Index - USD,...,None,None,None,None,None,0.000211,0.000190,None,Passive,ETF
2,Vanguard S&P 500 ETF,US9229083632,None,ETF,United States,United States,USD,ARCX,VOO,S&P 500 Daily Total Return Index - USD,...,None,None,None,None,None,0.000129,0.000095,None,Passive,ETF
3,iShares Core S&P 500 ETF,US4642872000,None,ETF,United States,"United States, Finland, Sweden, United Kingdom...",USD,ARCX,IVV,S&P 500 Daily Total Return Index - USD,...,None,None,None,None,None,0.000038,0.000021,None,Passive,ETF
4,ARMOR Core Risk-Managed ETF,US30151E4677,None,ETF,United States,United States,USD,ARCX,RMRC,NaN,...,None,None,None,None,None,NaN,NaN,None,Active,ETF
